# 🎙️ VoiceCloner Simple — Đọc file TXT

**4 bước đơn giản:**
1. Setup (chạy 1 lần, chờ ~5 phút)
2. Upload TXT
3. Upload giọng mẫu (tùy chọn)
4. Generate + tải về

In [ ]:
# ✅ BƯỚC 1: Setup (chạy 1 lần, chờ ~5 phút)
import os
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')
DRIVE_HF_CACHE = "/content/drive/MyDrive/viterbox_hf_cache"
os.makedirs(DRIVE_HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = DRIVE_HF_CACHE
os.environ['HUGGINGFACE_HUB_CACHE'] = DRIVE_HF_CACHE

# Cài uv
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1
os.environ['PATH'] = f"/root/.local/bin:{os.environ.get('PATH', '')}"

# Clone repo
import shutil
if os.path.exists('/content/VoiceCloner'):
    shutil.rmtree('/content/VoiceCloner')
!git clone https://github.com/kanazawahere/VoiceCloner.git /content/VoiceCloner --depth=1 --quiet
%cd /content/VoiceCloner

# Cài packages
import subprocess
cuda_ver = subprocess.getoutput(r"nvcc --version | grep 'release' | sed 's/.*release //' | sed 's/,.*//' | sed 's/\.//'").strip()
if not cuda_ver.isdigit():
    cuda_ver = "121"

print("⏳ Đang cài packages (2-3 phút)...")
!uv pip install --system torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu{cuda_ver} --quiet
!uv pip install --system -r requirements.txt --quiet
!uv pip install --system -e . --quiet

print("\n✅ Cài xong! Chạy cell tiếp để restart runtime.")

In [ ]:
# ⚠️ Restart runtime (tự động)
import os
os.kill(os.getpid(), 9)

In [ ]:
# 🔄 Load model (chạy sau khi restart)
import os
if 'HF_HOME' not in os.environ:
    os.environ['HF_HOME'] = "/content/drive/MyDrive/viterbox_hf_cache"
    os.environ['HUGGINGFACE_HUB_CACHE'] = "/content/drive/MyDrive/viterbox_hf_cache"

%cd /content/VoiceCloner

import torch
from viterbox import Viterbox

print("⏳ Đang load model (lần đầu ~5 phút, lần sau ~1 phút)...")
device = "cuda" if torch.cuda.is_available() else "cpu"
tts = Viterbox.from_pretrained(device)
print(f"✅ Model loaded on {device}")

---
## 📄 BƯỚC 2: Upload file TXT

In [ ]:
from google.colab import files

print("📤 Upload file TXT:")
uploaded = files.upload()
txt_file = list(uploaded.keys())[0]

with open(txt_file, 'r', encoding='utf-8') as f:
    text_content = f.read()

print(f"\n✅ Đã đọc {len(text_content)} ký tự")
print(f"Preview:\n{text_content[:200]}...")

---
## 🎤 BƯỚC 3: Upload giọng mẫu (tùy chọn)

In [ ]:
from google.colab import files

USE_VOICE_CLONING = True  # ← Đổi False để dùng giọng mặc định

if USE_VOICE_CLONING:
    print("📤 Upload file WAV (3-10 giây, sạch):")
    uploaded = files.upload()
    ref_audio = list(uploaded.keys())[0]
    print(f"✅ Giọng mẫu: {ref_audio}")
else:
    ref_audio = None
    print("✅ Dùng giọng mặc định")

---
## 🎛️ BƯỚC 4: Generate + điều chỉnh + tải về

In [ ]:
import librosa
import soundfile as sf
from IPython.display import Audio, display
from google.colab import files as colab_files

# ========== CÀI ĐẶT ==========
SPEED = 1.0      # 0.5=chậm 2x, 1.0=bình thường, 1.5=nhanh 1.5x
PITCH_SHIFT = 0  # -5=thấp hơn, 0=bình thường, +5=cao hơn
# ==============================

print("🎙️ Đang generate...")
audio = tts.generate(
    text=text_content,
    language="vi",
    audio_prompt=ref_audio,
    sentence_pause_ms=500
)

audio_np = audio.squeeze().cpu().numpy()
sr = 24000

# Điều chỉnh
if SPEED != 1.0:
    print(f"⚡ Tốc độ: {SPEED}x")
    audio_np = librosa.effects.time_stretch(audio_np, rate=SPEED)

if PITCH_SHIFT != 0:
    print(f"🎵 Cao độ: {PITCH_SHIFT:+d} semitones")
    audio_np = librosa.effects.pitch_shift(audio_np, sr=sr, n_steps=PITCH_SHIFT)

# Lưu
output_file = "/content/output.wav"
sf.write(output_file, audio_np, sr)

print(f"\n✅ Xong! Thời lượng: {len(audio_np)/sr:.1f}s")
display(Audio(output_file))

# Tải về
print("\n📥 Đang tải file về máy...")
colab_files.download(output_file)

---
## 📋 Hướng dẫn

### Điều chỉnh tốc độ/cao độ

Sửa trong **BƯỚC 4**:

```python
SPEED = 1.0
  0.5  = chậm 2x
  0.75 = chậm 1.3x
  1.0  = bình thường
  1.25 = nhanh 1.25x
  1.5  = nhanh 1.5x

PITCH_SHIFT = 0
  -5 = giọng thấp (nam)
   0 = bình thường
  +5 = giọng cao (nữ/trẻ em)
```

### Lưu vào Drive

Thêm vào cuối **BƯỚC 4**:
```python
import shutil
OUTPUT_DIR = "/content/drive/MyDrive/tts_outputs"
!mkdir -p {OUTPUT_DIR}
shutil.copy("/content/output.wav", f"{OUTPUT_DIR}/output.wav")
print(f"✅ Đã lưu vào {OUTPUT_DIR}")
```

### Tips
- Lần đầu chạy: BƯỚC 1 → Restart → Load model → BƯỚC 2-4
- Lần sau: Chỉ cần chạy BƯỚC 2-4 (model đã cache trong Drive)